In [ ]:
import os
os.environ["HSA_OVERRIDE_GFX_VERSION"]="11.0.0"
import re
from typing import List, Dict
from neo4j import GraphDatabase
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.cross_encoders import HuggingFaceCrossEncoder
from pydantic import BaseModel


# LLM_MODEL_NAME="SpeakLeash/bielik-11b-v2.3-instruct:Q8_0"
LLM_MODEL_NAME="jaro-jm/llm:latest"
DENSE_MODEL_NAME = "sdadas/stella-pl-retrieval"
RERANK_MODEL_NAME = "sdadas/polish-reranker-roberta-v3"
NEO4J_SPARSE_INDEX="node_fulltext_index"
NEO4J_DENSE_INDEX="node_embedding_index"
NEO4J_REL_DENSE_INDEX="rel_embedding_index"
NEO4J_REL_SPARSE_INDEX="rel_fulltext_index"
NEO4J_URL = "neo4j://localhost:7687"
NEO4J_USER = "neo4j"
NEO4J_PASSWORD = os.environ["NEO4J_PASSWORD"]

class Relation(BaseModel):
    head: str
    head_type: str
    relation: str
    tail: str
    tail_type: str

DENSE_MODEL = HuggingFaceEmbeddings(
    model=DENSE_MODEL_NAME,
    model_kwargs={"trust_remote_code": True, "device": "cpu"},
)

RERANKER = HuggingFaceCrossEncoder(
    model_name=RERANK_MODEL_NAME,
    model_kwargs={'trust_remote_code': True, 'device': 'cpu'}
)

In [ ]:
class GraphRepository:
    def __init__(self):
        self.driver = GraphDatabase.driver(NEO4J_URL, auth=(NEO4J_USER, NEO4J_PASSWORD))
        self.dense = DENSE_MODEL

    def _embed_text(self, text: str) -> list[float]:
        return self.dense.embed_query(text)

    def _embed_node(self, name: str, type_: str) -> list[float]:
        return self._embed_text(f"{type_}: {name}")

    def close(self):
        self.driver.close()

    @staticmethod
    def escape_lucene(text: str) -> str:
        return re.sub(r'([+\-!(){}[\]^"~*?:\\/])', r'\\\1', text)

    def update_chunk_graph(self, chunk_id: str, subgraph: List[Relation]):
        """
        Updates Neo4j with a "sub"-graph (the graph for a specific chunk).
        Each node has a type and a chunk_id list, relations between nodes are created by LM-service.
        """
        with self.driver.session() as session:
            for rel in subgraph:
                head_emb_calc = self._embed_node(rel.head, rel.head_type)
                tail_emb_calc = self._embed_node(rel.tail, rel.tail_type)
                rel_emb_calc = self._embed_text(rel.relation)
                session.execute_write(self._merge_relation, chunk_id, rel,
                                      head_emb_calc,
                                      tail_emb_calc,
                                      rel_emb_calc)

    def find_nearest_nodes(self, nodes: List[Dict[str, str]], top_k: int = 8) -> Dict[str, List[Dict]]:
        results = {}
        initial_k = top_k * 5
        hybrid_query = """
        CALL db.index.vector.queryNodes($vector_index, $k, $embedding)
        YIELD node, score
        WITH collect({node: node, score: score}) AS v

        CALL db.index.fulltext.queryNodes($fulltext_index, $text_query, {limit: $k})
        YIELD node, score
        WITH v, collect({node: node, score: score}) AS f

        WITH v + f AS allResults
        UNWIND allResults AS r
        WITH r.node AS n, max(r.score) AS finalScore
        RETURN elementId(n) AS id, n.name AS name, n.type AS type, finalScore AS score
        ORDER BY score DESC LIMIT $k
        """
        with self.driver.session() as session:
            for node in nodes:
                emb = self._embed_node(node['name'], node['type'])
                query_string = f"{node['type']}: {node['name']}"
                tokens = query_string.replace(":", "").split()
                text_query = " ".join([f"{self.escape_lucene(t)}~" for t in tokens])

                raw_res = session.run(
                    hybrid_query,
                    vector_index=NEO4J_DENSE_INDEX,
                    fulltext_index=NEO4J_SPARSE_INDEX,
                    k=initial_k,
                    embedding=emb,
                    text_query=text_query
                ).data()

                if not raw_res:
                    results[query_string] = []
                    continue

                pairs = [[query_string, f"{r['type']}: {r['name']}"] for r in raw_res]
                scores = RERANKER.score(pairs)
                for i, score in enumerate(scores):
                    raw_res[i]['rerank_score'] = float(score)

                final_nodes = sorted(raw_res, key=lambda x: x['rerank_score'], reverse=True)[:top_k]
                results[query_string] = final_nodes

        return results

    def find_nearest_relations(self, relations: List[str], top_k: int = 8) -> Dict[str, List[Dict]]:
        results = {}
        initial_k = top_k * 5
        hybrid_query = """
        CALL db.index.vector.queryRelationships($vector_index, $k, $embedding)
        YIELD relationship, score
        WITH collect({rel: relationship, score: score}) AS v

        CALL db.index.fulltext.queryRelationships($fulltext_index, $text_query, {limit: $k})
        YIELD relationship, score
        WITH v, collect({rel: relationship, score: score}) AS f

        WITH v + f AS allResults
        UNWIND allResults AS r
        WITH r.rel AS rel, max(r.score) AS finalScore
        RETURN elementId(startNode(rel)) AS head_id,
               startNode(rel).name AS head,
               startNode(rel).type AS head_type,
               rel.type AS relation_name,
               elementId(endNode(rel)) AS tail_id,
               endNode(rel).name AS tail,
               endNode(rel).type AS tail_type,
               finalScore AS score
        ORDER BY score DESC LIMIT $k
        """

        with self.driver.session() as session:
            for query_text in relations:
                emb = self._embed_text(query_text)
                tokens = query_text.split()
                text_query = " ".join([f"{self.escape_lucene(t)}~" for t in tokens])

                raw_res = session.run(
                    hybrid_query,
                    vector_index=NEO4J_REL_DENSE_INDEX,
                    fulltext_index=NEO4J_REL_SPARSE_INDEX,
                    k=initial_k,
                    embedding=emb,
                    text_query=text_query
                ).data()

                if not raw_res:
                    results[query_text] = []
                    continue

                pairs = []
                for r in raw_res:
                    doc_text = f"{r['head']} ({r['head_type']}) - {r['relation_name']} -> {r['tail']} ({r['tail_type']})"
                    pairs.append([query_text, doc_text])

                scores = RERANKER.score(pairs)
                for i, score in enumerate(scores):
                    raw_res[i]['rerank_score'] = float(score)

                final_rels = sorted(raw_res, key=lambda x: x['rerank_score'], reverse=True)[:top_k]
                results[query_text] = final_rels

        return results

    def _expand_subgraph(self, start_node_ids: list, hops: int, limit: int) -> List[str]:
        """
        Expands the context by 'hops' hops from the given start nodes.
        """
        if not start_node_ids or hops < 1:
            return []

        # 'Traversing' query
        # DISTINCT r, so relations are not duplicated if the graph has cycles
        cypher = f"""
            MATCH (start)
            WHERE elementId(start) IN $ids
            MATCH path = (start)-[r*1..{hops}]-(end)
            UNWIND relationships(path) AS rel
            WITH DISTINCT rel
            RETURN startNode(rel).name AS head,
                   startNode(rel).type AS head_type,
                   rel.type AS relation,
                   endNode(rel).name AS tail,
                   endNode(rel).type AS tail_type
            LIMIT $limit
        """

        expanded_lines = []
        with self.driver.session() as session:
            results = session.run(cypher, ids=start_node_ids, limit=limit).data()

            for res in results:
                line = (f"({res['head']}: {res['head_type']}) "
                        f"--[{res['relation']}]--> "
                        f"({res['tail']}: {res['tail_type']})")
                expanded_lines.append(line)

        return expanded_lines

    def build_nearest_subgraphs(self, nearest_nodes: dict, nearest_relations: list,
                                max_triples: int = 5, max_nodes: int = 8, hops: int = 2) -> str:

        context_parts = []
        collected_ids = set()

        # Relations handling
        if nearest_relations:
            sorted_rels = sorted(
                nearest_relations,
                key=lambda r: r.get("rerank_score", 0),
                reverse=True
            )[:max_triples]

            lines = []
            for idx, rel in enumerate(sorted_rels):
                if 'head_id' in rel: collected_ids.add(rel['head_id'])
                if 'tail_id' in rel: collected_ids.add(rel['tail_id'])

                head = rel.get("head", "UNKNOWN")
                tail = rel.get("tail", "UNKNOWN")
                rel_name = rel.get("relation_name", rel.get("relation", "UNKNOWN"))

                head_info = rel.get("head_type")
                tail_info = rel.get("tail_type")

                lines.append(
                    f"Znaleziona Relacja {idx}: "
                    f"{head} ({head_info}) --[{rel_name}]--> {tail} ({tail_info})"
                )

            if lines:
                context_parts.append("BEZPOŚREDNIE TRAFIENIA (RELACJE):\n" + "\n".join(lines))

        # Nodes handling
        if nearest_nodes:
            all_nodes = []
            for _, items in nearest_nodes.items():
                all_nodes.extend(items)

            # Deduplication by name and type
            seen = set()
            unique_nodes = []
            for n in all_nodes:
                nid = n.get('id')
                if nid and nid not in seen:
                    seen.add(nid)
                    unique_nodes.append(n)
                    collected_ids.add(nid)

            unique_nodes = sorted(unique_nodes, key=lambda n: n.get("rerank_score", 0), reverse=True)[:max_nodes]

            lines = []
            for idx, n in enumerate(unique_nodes):
                lines.append(f"Znaleziony Byt {idx}: {n.get('name')} (Typ: {n.get('type')})")

            if lines:
                context_parts.append("BEZPOŚREDNIE TRAFIENIA (BYTY):\n" + "\n".join(lines))

        # 'hop' logic (Graph Expansion)
        if hops > 0 and collected_ids:
            neighborhood = self._expand_subgraph(list(collected_ids), hops=hops, limit=30)

            if neighborhood:
                context_parts.append(
                    f"\nSZERSZY KONTEKST (Sąsiedztwo {hops}-hop):\n" +
                    "Logika: Relacje prowadzące od/do znalezionych wyżej elementów.\n" +
                    "\n".join(neighborhood)
                )

        if not context_parts:
            return ""

        return "\n\n".join(context_parts)

    # deprecated
    def _find_node_info(self, nearest_nodes: dict, node_name: str, node_type: str | None = None) -> str | None:
        """
        Looks for node_name in the nearest_nodes results and returns compressed info (e.g. 'type, score').
        nearest_nodes has keys in the form "Type: Name" -> list of results.
        We search the lists and if a match is found, return the type and avg score.
        """
        # if nearest_nodes is empty -> None
        if not nearest_nodes:
            return None

        keys_to_check = []
        if node_type:
            keys_to_check.append(f"{node_type}: {node_name}")
        keys_to_check.append(node_name)

        for key in keys_to_check:
            if key in nearest_nodes:
                items = nearest_nodes[key]
                if items:
                    t = items[0].get("type", "")
                    s = items[0].get("score", None)
                    return f"{t}{', score=' + ('{:.4f}'.format(s) if s is not None else 'n/a')}"

        for items in nearest_nodes.values():
            for it in items:
                if it.get("name") == node_name:
                    t = it.get("type", "")
                    s = it.get("score", None)
                    return f"{t}{', score=' + ('{:.4f}'.format(s) if s is not None else 'n/a')}"

        return None


    @staticmethod
    def _merge_relation(tx, chunk_id: str, rel: Relation, head_emb: list[float], tail_emb: list[float], rel_emb: list[float]):
        """
        1. Creates or updates (upsert) head and tail (adds chunk_id to the list if missing)
        2. Creates the relation between them
        """
        # MERGE head node
        tx.run(
            """
            MERGE (h:Node {name: $head_name, type: $head_type})
            ON CREATE SET
                h.chunk_ids = [$chunk_id],
                h.embedding = $embedding,
                h.text = $text
            ON MATCH SET h.chunk_ids = CASE WHEN NOT $chunk_id IN h.chunk_ids THEN h.chunk_ids + $chunk_id ELSE h.chunk_ids END
            """,
            head_name=rel.head,
            head_type=rel.head_type,
            chunk_id=chunk_id,
            embedding=head_emb,
            text=f"{rel.head_type}: {rel.head}"
        )

        # MERGE tail node
        tx.run(
            """
            MERGE (t:Node {name: $tail_name, type: $tail_type})
            ON CREATE SET
                t.chunk_ids = [$chunk_id],
                t.embedding = $embedding,
                t.text = $text
            ON MATCH SET t.chunk_ids = CASE WHEN NOT $chunk_id IN t.chunk_ids THEN t.chunk_ids + $chunk_id ELSE t.chunk_ids END
            """,
            tail_name=rel.tail,
            tail_type=rel.tail_type,
            chunk_id=chunk_id,
            embedding=tail_emb,
            text=f"{rel.tail_type}: {rel.tail}"
        )

        # MERGE relation
        tx.run(
            """
            MATCH (h:Node {name: $head_name, type: $head_type})
            MATCH (t:Node {name: $tail_name, type: $tail_type})
            MERGE (h)-[r:RELATED]->(t)
            ON CREATE SET
                r.chunk_ids = [$chunk_id],
                r.type = $original_rel_type,
                r.embedding = $rel_embedding
            ON MATCH SET r.chunk_ids = CASE WHEN NOT $chunk_id IN r.chunk_ids THEN r.chunk_ids + $chunk_id ELSE r.chunk_ids END
            """,
            head_name=rel.head,
            head_type=rel.head_type,
            tail_name=rel.tail,
            tail_type=rel.tail_type,
            chunk_id=chunk_id,
            original_rel_type=rel.relation,
            rel_embedding=rel_emb
        )

    def test(self):
        """
        Diagnostic test of embedding + Neo4j.
        Run manually, not for production.
        """

        vector_only_query = """
        CALL db.index.vector.queryNodes(
            $vector_index,
            5,
            $embedding
        )
        YIELD node, score
        RETURN node.name AS name, node.type AS type, score
        ORDER BY score DESC
        """

        hybrid_query = """
        CALL db.index.vector.queryNodes($vector_index, 5, $embedding)
        YIELD node, score
        WITH collect({node: node, score: score}) AS v

        CALL db.index.fulltext.queryNodes($fulltext_index, $text_query, {limit: 5})
        YIELD node, score
        WITH v, collect({node: node, score: score}) AS f

        UNWIND (v + f) AS r
        WITH DISTINCT r.node AS node, r.score AS score
        RETURN node.name AS name, node.type AS type, score
        ORDER BY score DESC
        """

        queries = [
            "shahed",
            "dron shahed",
            "adwokat"
        ]

        with self.driver.session() as session:
            print("\n=== TEST 1: round-trip (sanity check) ===")
            rt = session.run(
                """
                MATCH (n:Node)
                WITH n LIMIT 1
                CALL db.index.vector.queryNodes(
                    $vector_index,
                    1,
                    n.embedding
                )
                YIELD node, score
                RETURN n.name AS original, node.name AS returned, score
                """,
                vector_index=NEO4J_DENSE_INDEX
            ).data()
            print(rt)

            for q in queries:
                print(f"\n=== TEST 2: vector only | query = '{q}' ===")
                emb = self._embed_text(q)
                res_vec = session.run(
                    vector_only_query,
                    vector_index=NEO4J_DENSE_INDEX,
                    embedding=emb
                ).data()
                for r in res_vec:
                    print(r)

                print(f"\n=== TEST 3: hybrid | query = '{q}' ===")
                tokens = q.split()
                fuzzy = " ".join(f"{self.escape_lucene(t)}~" for t in tokens)

                res_hybrid = session.run(
                    hybrid_query,
                    vector_index=NEO4J_DENSE_INDEX,
                    fulltext_index=NEO4J_SPARSE_INDEX,
                    embedding=emb,
                    text_query=fuzzy
                ).data()
                for r in res_hybrid:
                    print(r)


g = GraphRepository()
try:
    print("--- DEMO: RAG with graph expansion (Hops) ---")

    search_relations = ["GRANICZY Z", "SĄSIADUJE"]
    search_nodes = [{"name": "Polska", "type": "Country"}]

    print(f"\n[1] Searching relations: {search_relations}")
    rels_map = g.find_nearest_relations(search_relations, top_k=5)

    all_rels_flat = []
    for r_list in rels_map.values():
        all_rels_flat.extend(r_list)

    print(f"\n[2] Searching nodes: {search_nodes}")
    nodes_map = g.find_nearest_nodes(search_nodes, top_k=3)

    print("\n[3] Generating context with expansion (hops=2)...")
    context = g.build_nearest_subgraphs(
        nearest_nodes=nodes_map,
        nearest_relations=all_rels_flat,
        max_triples=5,
        max_nodes=5,
        hops=2
    )

    print("="*60)
    print("GENERATED CONTEXT FOR LLM:")
    print("="*60)
    print(context)
    print("="*60)

except Exception as e:
    print(f"ERROR: {e}")
    import traceback
    traceback.print_exc()
finally:
    g.close()

In [ ]:
subgraph = [
    # --- SCENARIO 1: Entity Resolution (Different names of the same entity) ---
    # Expected behavior: Merge the nodes "Stany Zjednoczone" and "USA" into one (e.g. "USA").
    Relation(head="Stany Zjednoczone", head_type="Country", relation="REALIZUJE", tail="Strategia odcięcia Chin", tail_type="Policy"),
    Relation(head="USA", head_type="Country", relation="DZIAŁANIE", tail="Globalna strategia", tail_type="Policy"),

    # --- SCENARIO 2: Relation Synonyms ---
    # Expected behavior: Merge the relations "ATAKUJE" and "PROWADZI AGRESJĘ" into one (e.g. "ATAKUJE").
    Relation(head="Rosja", head_type="Country", relation="ATAKUJE", tail="Ukraina", tail_type="Country"),
    Relation(head="Federacja Rosyjska", head_type="Country", relation="PROWADZI AGRESJĘ", tail="Ukraina", tail_type="Country"),

    # --- SCENARIO 3: Ambiguity of first and last names (Full name vs Last name) ---
    # Expected behavior: Merge "Hołownia" with "Szymon Hołownia".
    Relation(head="Polska 2050", head_type="Party", relation="LIDER", tail="Szymon Hołownia", tail_type="Person"),
    Relation(head="Hołownia", head_type="Person", relation="DECYZJA", tail="Start w wyborach", tail_type="Event"),

    # --- SCENARIO 4: Acronym Resolution (Expanding abbreviations) ---
    # Expected behavior: Link ECA with the full name of the region.
    Relation(head="ECA", head_type="Region", relation="PEŁNA_NAZWA", tail="Europa i Azja Środkowa", tail_type="Region"),
    Relation(head="Europa i Azja Środkowa", head_type="Region", relation="WZROST_PKB", tail="2.4 proc.", tail_type="EconomicIndicator"),

    # --- SCENARIO 5: Extraction errors (Two entities in one node) ---
    # Expected behavior: The agent should split the node "Kamiński i Wąsik" into two separate ones and duplicate the relation.
    Relation(head="Wojciech Łączewski", head_type="Judge", relation="SKAZAŁ", tail="Kamiński i Wąsik", tail_type="Person"),

    # --- SCENARIO 6: Roles changing over time (Same person, different roles) ---
    # Expected behavior: Keep a single node "Wojciech Łączewski", but preserve both relations (historical and current).
    Relation(head="Wojciech Łączewski", head_type="Person", relation="BYŁA_FUNKCJA", tail="Sędzia", tail_type="Role"),
    Relation(head="Wojciech Łączewski", head_type="Person", relation="OBECNA_FUNKCJA", tail="Adwokat", tail_type="Role"),

    # --- SCENARIO 7: Level of detail (Granularity) ---
    # Expected behavior: Link the generic "Port" with the specific name "Port Haller".
    Relation(head="PiS", head_type="Party", relation="PLANUJE", tail="Port Haller", tail_type="Project"),
    Relation(head="Kaczyński", head_type="Person", relation="MÓWI O", tail="Nowy port w Choczewie", tail_type="Project"),

    # --- SCENARIO 8: Typos and minor differences (Fuzzy Matching) ---
    # Expected behavior: Merge "Wołodymyr Zełenski" and "WłodymyrZełenski" (OCR/text error).
    Relation(head="Wołodymyr Zełenski", head_type="Person", relation="OGŁASZA", tail="Atak dronów", tail_type="Event"),
    Relation(head="WłodymyrZełenski", head_type="Person", relation="KOMENTUJE", tail="Wsparcie USA", tail_type="Action"),

    # --- SCENARIO 9: Conflicting types (Type Conflict) ---
    # Expected behavior: Settle on one type for "Polska 2050" (is it Organization or PoliticalParty?).
    Relation(head="Polska 2050", head_type="Organization", relation="ORGANIZUJE", tail="Głosowanie online", tail_type="Event"),
    Relation(head="Polska 2050", head_type="PoliticalParty", relation="PROBLEMY", tail="Atak hakerski", tail_type="Event"),

    # --- SCENARIO 10: Transitive relations / Redundancy ---
    # Expected behavior: Decide whether to keep all of them or simplify the graph.
    Relation(head="Shahed", head_type="Weapon", relation="JEST_TYPEM", tail="Dron", tail_type="WeaponType"),
    Relation(head="Rosja", head_type="Country", relation="UŻYWA", tail="Shahed", tail_type="Weapon"),
    Relation(head="Rosja", head_type="Country", relation="UŻYWA", tail="Dron", tail_type="WeaponType"), # Potentially redundant

    # --- ADDITIONAL BACKGROUND RELATIONS (Context) ---
    Relation(head="Wojciech Czuchnowski", head_type="Journalist", relation="PRACUJE_DLA", tail="Gazeta Wyborcza", tail_type="Media"),
    Relation(head="Izrael", head_type="Country", relation="SOJUSZNIK", tail="USA", tail_type="Country"),
    Relation(head="Chiny", head_type="Country", relation="IMPORTUJE", tail="Węglowodory", tail_type="Resource"),
    Relation(head="Katarzyna Pełczyńska-Nałęcz", head_type="Person", relation="KANDYDUJE", tail="Przewodniczący partii", tail_type="Role"),
    Relation(head="Paulina Hennig-Kloska", head_type="Person", relation="KANDYDUJE", tail="Przewodniczący partii", tail_type="Role"),

    # --- CHAIN 1: Incident path (Cause -> Effect -> Verification) [5 steps] ---
    # Tests: The ability to reconstruct the chronology of events.

    # 1. Perpetrator -> Action
    Relation(head="Izraelski trener", head_type="Person", relation="ZAATAKOWAŁ", tail="Sędzia zawodów", tail_type="Person"),
    # 2. Victim -> Context
    Relation(head="Sędzia zawodów", head_type="Person", relation="PROWADZIŁ", tail="Turniej judo w Bielsku-Białej", tail_type="Event"),
    # 3. Context -> Organizer's reaction
    Relation(head="Turniej judo w Bielsku-Białej", head_type="Event", relation="ZAKOŃCZONY_PRZEZ", tail="Wykluczenie drużyny", tail_type="Action"),
    # 4. Reaction -> External verification
    Relation(head="Wykluczenie drużyny", head_type="Action", relation="ZWERYFIKOWANE_PRZEZ", tail="Policja", tail_type="Organization"),
    # 5. Verification -> Result
    Relation(head="Policja", head_type="Organization", relation="POTWIERDZA", tail="Wersja organizatorów", tail_type="Statement"),


    # --- CHAIN 2: Diplomatic path (Accusation -> Reaction -> Source of truth) [4 steps] ---
    # Tests: Name resolution (Maciej Wewiór = Rzecznik MSZ) and the hierarchy of statements.

    # 1. Actor A -> Action
    Relation(head="Ambasada Izraela", head_type="Organization", relation="OSKARŻA_O", tail="Antysemickie zachowanie", tail_type="Concept"),
    # 2. Actor B -> Reaction to Actor A
    Relation(head="Maciej Wewiór", head_type="Person", relation="ODPOWIADA_NA", tail="Oskarżenia Ambasady", tail_type="Statement"),
    # 3. Actor B -> Role
    Relation(head="Maciej Wewiór", head_type="Person", relation="PEŁNI_FUNKCJĘ", tail="Rzecznik MSZ", tail_type="Role"),
    # 4. Actor B -> Citing evidence
    Relation(head="Rzecznik MSZ", head_type="Role", relation="PRZYTACZA", tail="Oświadczenie Polskiego Związku Judo", tail_type="Document"),


    # --- CHAIN 3: Narrative conflict (Narrative 1 vs Narrative 2) [3 steps] ---
    # Tests: Whether the Agent notices the contradiction in the graph (Attack ON the athletes vs Attack BY the coach).

    # "Fake News" path / Ambassador's version
    Relation(head="Ambasador Izraela", head_type="Person", relation="TWEETUJE_O", tail="Atak na zawodników z Izraela", tail_type="Event"),

    # Linking node (Same event, different interpretation)
    Relation(head="Atak na zawodników z Izraela", head_type="Event", relation="DOTYCZY_TEGO_SAMEGO_CO", tail="Incydent w Bielsku-Białej", tail_type="Event"),

    # "Fact Check" path / Version of MSZ and the Police
    Relation(head="Incydent w Bielsku-Białej", head_type="Event", relation="SPOWODOWANY_PRZEZ", tail="Agresja izraelskiego trenera", tail_type="Event"),


    # --- EXTRA: Entity Resolution in context (Synonyms) ---
    Relation(head="MSZ", head_type="Organization", relation="MA_RZECZNIKA", tail="Maciej Wewiór", tail_type="Person"),
    Relation(head="Ministerstwo Spraw Zagranicznych", head_type="Organization", relation="WYDAJE", tail="Komunikat dyplomatyczny", tail_type="Document")
]

g = GraphRepository()
try:
    g.update_chunk_graph("chunk_test", subgraph)
finally:
    g.close()

TODO: Code for the langgraph agent logic, the agent will be run:
 - on a live neo4j graph, e.g. once a day on the whole graph at midnight,
 - when adding a subgraph to the main graph (the graph from a given chunk).

1. List of actions the langgraph agent will be able to perform:
    - Searching for similar nodes / relations (the agent must 'see' the graph to be able to act)
    - merging nodes,
    - merging relations,
    - deleting nodes,
    - deleting relations,
    - updating nodes,
    - updating relations,
    - adding nodes,
    - adding relations,
    - splitting nodes,
    - splitting relations,
    - no action / do nothing (when it is already ok, no action is required or it does not know what to do).

2. Methods in the GraphRepository class matching the actions above.

3. Modify the langgraph sample code below so that it deduplicates / cleans the graph stored in neo4j. Use GraphRepository - find_nearest_nodes and find_nearest_relations. The agent methods must use the GraphRepository methods.

4. Agent logic when adding a subgraph to the main graph:
    - A subgraph is to be inserted as a set of triples (objects of the Relation class). Such a subgraph from a chunk is inserted into the main graph in neo4j.
    - Start by finding in the main graph the nodes nearest to the nodes of the chunk subgraph being inserted (find_nearest_nodes).
    - For the nodes found in the main graph, find all the nodes 'surrounding' them (connected by a relation) - the _expand_subgraph method and the hops logic. This gives us subgraphs from the main graph.
    - The subgraphs obtained this way from the main graph are compared with the chunk subgraph. The llm decides to perform one of the actions from point 1, e.g.:
        - when there are no similar nodes in the main-graph subgraphs relative to the chunk subgraph, we perform the "no action / do nothing" action
        - when the main-graph subgraphs contain the node "Putin" and the chunk subgraph has "Vladimir Putin", the main-graph node should be unified with the "updating nodes" action before adding nodes and relations to the main graph